# Does downside beta explain the cross-section of stock returns?
**CFM 301 – Project 2 (Finding and Testing a Factor)** · S&P 500 constituents, monthly, 2000–2025

**Question.** Do investors earn a higher return for holding stocks that fall harder *when the market is already falling* (high downside beta), beyond what ordinary CAPM beta already explains? Audience: quantitative fund managers / pension funds deciding whether to tilt toward (or away from) downside-beta exposure.

**Benchmark and models.** CAPM is the base benchmark. All three regressions are estimated month by month on the cross-section of stocks (Fama–MacBeth), on the *same* stock-months, so their fit is directly comparable:

| | Specification (month *t+1* cross-section) |
|---|---|
| **CAPM benchmark** | $R^e_{i,t+1} = a + \gamma_\beta\,\beta_{i,t} + \epsilon_{i,t+1}$ |
| **Model 1: downside beta only** | $R^e_{i,t+1} = a + \gamma_D\,\beta^-_{i,t} + \epsilon_{i,t+1}$ |
| **Model 2: CAPM beta + downside beta** | $R^e_{i,t+1} = a + \gamma_\beta\,\beta_{i,t} + \gamma_D\,\beta^-_{i,t} + \epsilon_{i,t+1}$ |

**Variables.**
- $R^e_{i,t+1} = R_{i,t+1} - R_{f,t+1}$: stock $i$'s total return (dividends included, supplied `total_return`) minus the risk-free rate, in the month **after** the betas are measured.
- $\beta_{i,t}$: ordinary beta = Cov(R_i−R_f, R_m−R_f) / Var(R_m−R_f) over the 60 months ending in month *t*.
- $\beta^-_{i,t}$: downside beta = the same ratio computed only over the months in that window where R_m−R_f < 0.
- $\gamma_D$ is the coefficient of interest: $H_0:\gamma_D = 0$. Under a risk-based story $\gamma_D > 0$ (investors are paid for bad-state exposure).

**What changed relative to the old notebook.** (1) Betas are now **lagged one month** – the old regressions matched the beta ending in month *t* to the return in month *t*, so the beta window contained the return being explained (look-ahead). (2) Per-stock time-series regressions on a slowly moving beta were replaced by **Fama–MacBeth cross-sectional regressions with Newey–West standard errors**, the standard test for a beta/characteristic premium. (3) CAPM is an explicit benchmark, with identical samples across specifications. (4) Sensitivity analysis, a bad-times check, and cleaning counts were added.

## 1. Setup and configuration
All tunable choices live in this cell. Paths are relative; the three CSV files are looked for in `./data`, `./`, or `./CAPM Factors`.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:  # running as a plain script
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

# ---- Beta estimation -------------------------------------------------------
WINDOW_MONTHS = 60        # trailing window, includes month t, excludes the future
MIN_DOWN_OBS = 12         # minimum down-market months for a downside beta
MIN_BETA_OBS = 36         # minimum valid months for an ordinary beta

# ---- Regression sample and inference ---------------------------------------
REG_START, REG_END = None, None   # None = use all available return months
WINSOR_LIMITS = (0.01, 0.99)      # cross-sectional, within each month
NW_LAGS = 6                       # Newey-West lags for Fama-MacBeth t-stats
MIN_STOCKS_PER_MONTH = 30         # skip months with fewer stocks
SIGNIFICANCE_LEVEL = 0.05

# ---- Files -----------------------------------------------------------------
def find_data_dir():
    for candidate in [Path.cwd() / "data", Path.cwd(), Path.cwd() / "CAPM Factors"]:
        if (candidate / "sp500_stocks_2000_2025.csv").is_file():
            return candidate
    raise FileNotFoundError("Could not find the input CSV files.")

DATA_DIR = find_data_dir()
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

## 2. Statistical helpers
Student-t p-values (no SciPy needed), an OLS routine with Newey–West (HAC) standard errors, the Fama–MacBeth estimator, and a pooled OLS with standard errors clustered by month (used only as a cross-check).

In [ ]:
# ---- Student-t distribution (regularized incomplete beta) -------------------
def _beta_cf(a, b, x):
    tiny, qab, qap, qam = 1e-300, a + b, a + 1, a - 1
    c, d = 1.0, 1.0 - qab * x / qap
    d = 1.0 / (d if abs(d) > tiny else tiny)
    h = d
    for i in range(1, 201):
        m2 = 2 * i
        for aa in (i * (b - i) * x / ((qam + m2) * (a + m2)),
                   -(a + i) * (qab + i) * x / ((a + m2) * (qap + m2))):
            d = 1.0 + aa * d
            d = d if abs(d) > tiny else tiny
            c = 1.0 + aa / c
            c = c if abs(c) > tiny else tiny
            d = 1.0 / d
            delta = d * c
            h *= delta
        if abs(delta - 1.0) < 3e-14:
            return h
    raise ArithmeticError("Incomplete-beta evaluation did not converge.")


def _regularized_beta(x, a, b):
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    front = math.exp(math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
                     + a * math.log(x) + b * math.log1p(-x))
    if x < (a + 1) / (a + b + 2):
        return front * _beta_cf(a, b, x) / a
    return 1 - front * _beta_cf(b, a, 1 - x) / b


def t_pvalue(t_stat, df):
    # two-sided p-value
    return _regularized_beta(df / (df + t_stat * t_stat), df / 2, 0.5)


def t_critical(alpha, df):
    low, high = 0.0, 100.0
    for _ in range(80):
        mid = (low + high) / 2
        low, high = (mid, high) if t_pvalue(mid, df) > alpha else (low, mid)
    return (low + high) / 2


# ---- OLS with Newey-West (HAC) standard errors ------------------------------
def ols_hac(y, X, lags):
    y, X = np.asarray(y, float), np.asarray(X, float)
    T, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ (X.T @ y)
    u = y - X @ b
    xu = X * u[:, None]
    S = xu.T @ xu
    for lag in range(1, lags + 1):
        w = 1 - lag / (lags + 1)          # Bartlett kernel
        G = xu[lag:].T @ xu[:-lag]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.diag(V))
    t = b / se
    df = T - k
    p = np.array([t_pvalue(abs(v), df) for v in t])
    return b, se, t, p, df


def coef_table(names, b, se, t, p, df):
    crit = t_critical(SIGNIFICANCE_LEVEL, df)
    return pd.DataFrame({"coef": b, "se": se, "t": t, "p": p,
                         "ci_low": b - crit * se, "ci_high": b + crit * se}, index=names)


# ---- Fama-MacBeth ------------------------------------------------------------
def fama_macbeth(panel, regressors, lags=None):
    # One cross-sectional OLS per month; then time-series mean of the slopes with
    # Newey-West standard errors. Also stores each month's R-squared / adjusted R-squared.
    lags = NW_LAGS if lags is None else lags
    months, slopes, fit = [], [], []
    for month, g in panel.groupby("return_month", sort=True):
        n = len(g)
        if n < MIN_STOCKS_PER_MONTH:
            continue
        X = np.column_stack([np.ones(n), g[regressors].to_numpy(float)])
        y = g["excess_return"].to_numpy(float)
        b = np.linalg.lstsq(X, y, rcond=None)[0]
        u = y - X @ b
        sst = float(((y - y.mean()) ** 2).sum())
        r2 = 1 - float(u @ u) / sst
        k = X.shape[1]
        months.append(month)
        slopes.append(b)
        fit.append((r2, 1 - (1 - r2) * (n - 1) / (n - k), n))
    idx = pd.DatetimeIndex(months, name="return_month")
    gamma = pd.DataFrame(slopes, index=idx, columns=["const", *regressors])
    stats = pd.DataFrame(fit, index=idx, columns=["r2", "adj_r2", "n_stocks"])
    one = np.ones((len(gamma), 1))
    rows = [ols_hac(gamma[c].to_numpy(), one, lags) for c in gamma.columns]
    summary = coef_table(
        gamma.columns,
        np.array([r[0][0] for r in rows]), np.array([r[1][0] for r in rows]),
        np.array([r[2][0] for r in rows]), np.array([r[3][0] for r in rows]),
        len(gamma) - 1,
    )
    return {"gamma": gamma, "stats": stats, "summary": summary, "regressors": regressors}


# ---- Pooled OLS, SE clustered by month (cross-check only) ----------------------
def pooled_ols_cluster(panel, regressors):
    X = np.column_stack([np.ones(len(panel)), panel[regressors].to_numpy(float)])
    y = panel["excess_return"].to_numpy(float)
    n, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ (X.T @ y)
    u = y - X @ b
    codes, _ = pd.factorize(panel["return_month"])
    G = codes.max() + 1
    scores = np.zeros((G, k))
    np.add.at(scores, codes, X * u[:, None])
    V = (G / (G - 1)) * ((n - 1) / (n - k)) * XtX_inv @ (scores.T @ scores) @ XtX_inv
    se = np.sqrt(np.diag(V))
    t = b / se
    p = np.array([t_pvalue(abs(v), G - 1) for v in t])
    r2 = 1 - float(u @ u) / float(((y - y.mean()) ** 2).sum())
    return coef_table(["const", *regressors], b, se, t, p, G - 1), r2, n

## 3. Load and clean the data
**Returns.** The supplied `total_return` is the monthly total return (dividends included, split-adjusted); it is used as is – it is never recomputed from the raw `price` column, which is not split-adjusted. Units are decimals (0.05 = 5%) in all three files. Dates are month-ends, and `permno` identifies securities (tickers can be reused).

**Cleaning steps** (each step's count is logged below): parse and validate dates/identifiers; drop exact duplicate PERMNO-month rows (error if duplicates conflict); drop missing returns (never set to zero); inspect extreme returns (kept – see note below).

**Delisting returns.** The file contains total returns but no separate delisting-return field, so no extra delisting adjustment is possible; the last return of a delisted stock is whatever the file reports.

**Universe note.** The file holds S&P 500 members in the months they were in the index (about 500 per month). Stocks therefore enter with no pre-membership history, which limits beta windows for recent additions and introduces some selection into the sample.

In [ ]:
# ---- Market and risk-free rate -------------------------------------------------
market = pd.read_csv(DATA_DIR / "market_returns_2000_2025.csv", parse_dates=["date"])
risk_free = pd.read_csv(DATA_DIR / "risk_free_rate_2000_2025.csv", parse_dates=["date"])
monthly = market.merge(risk_free, on="date", how="outer", validate="one_to_one", indicator=True)
if (monthly["_merge"] != "both").any():
    raise ValueError("Market and risk-free dates do not match.")
if monthly[["market_return", "rf_1m"]].isna().any().any():
    raise ValueError("Market return or risk-free rate has missing values.")

factors = pd.DataFrame(
    {"Mkt-RF": (monthly["market_return"] - monthly["rf_1m"]).to_numpy(),
     "RF": monthly["rf_1m"].to_numpy()},
    index=pd.DatetimeIndex(monthly["date"], name="date"),
).sort_index()
calendar = factors.index
if not calendar.equals(pd.date_range(calendar.min(), calendar.max(), freq=pd.offsets.MonthEnd())):
    raise ValueError("Market data must have one observation per month-end with no gaps.")

# ---- Stock returns --------------------------------------------------------------
stocks = pd.read_csv(DATA_DIR / "sp500_stocks_2000_2025.csv", parse_dates=["date"])
required = {"date", "permno", "ticker", "company_name", "total_return"}
if required - set(stocks.columns):
    raise ValueError(f"Missing columns: {sorted(required - set(stocks.columns))}")
if stocks[["date", "permno"]].isna().any().any():
    raise ValueError("Missing dates or PERMNOs.")
if not stocks["date"].dt.is_month_end.all():
    raise ValueError("Expected month-end dates in the stock file.")
stocks["total_return"] = pd.to_numeric(stocks["total_return"], errors="raise")

log = [("Raw stock-month rows", len(stocks), len(stocks))]

dup_mask = stocks.duplicated(["permno", "date"], keep=False)
if dup_mask.any() and stocks[dup_mask].groupby(["permno", "date"]).nunique(dropna=False).gt(1).any(axis=None):
    raise ValueError("Conflicting duplicate PERMNO/date rows.")
n_dup = int(stocks.duplicated(["permno", "date"]).sum())
stocks = stocks.drop_duplicates(["permno", "date"])
log.append(("Duplicate PERMNO-month rows dropped", n_dup, len(stocks)))

n_missing = int(stocks["total_return"].isna().sum())
stocks = stocks.dropna(subset=["total_return"])
log.append(("Missing total_return dropped (not set to 0)", n_missing, len(stocks)))

n_extreme = int((stocks["total_return"].abs() > 1.0).sum())
log.append(("Returns with |R| > 100% flagged for inspection (kept)", n_extreme, len(stocks)))

cleaning_log = pd.DataFrame(log, columns=["step", "rows_affected", "rows_remaining"])
display(cleaning_log)

print("Largest absolute returns, listed for inspection (kept: they cluster in crisis periods such as 2002, 2008-09, 2020, 2023; verify any you doubt):")
extremes = stocks.reindex(stocks["total_return"].abs().sort_values(ascending=False).index).head(8)
display(extremes[["date", "permno", "ticker", "company_name", "total_return"]])

# ---- Wide panel of RAW (un-winsorized) total returns ----------------------------
# Betas are estimated on raw returns: valid extremes are real observations.
ret_wide = stocks.pivot(index="date", columns="permno", values="total_return").reindex(calendar)
permnos = ret_wide.columns.to_numpy()
excess_wide = ret_wide.sub(factors["RF"], axis=0)   # R_i - R_f, aligned by month
print(f"{len(permnos)} securities, {len(calendar)} months ({calendar[0]:%Y-%m} to {calendar[-1]:%Y-%m})")

## 4. Rolling ordinary and downside betas
For each stock and month-end *t*, using the 60 calendar months ending in *t* (e.g. the December 2020 estimate uses January 2016 – December 2020; nothing after *t* is used):

- **Ordinary beta:** Cov(R_i−R_f, R_m−R_f) / Var(R_m−R_f) over all months in the window with both returns present (at least `MIN_BETA_OBS`).
- **Downside beta:** the same ratio but keeping only months with R_m−R_f **strictly < 0**; the same retained months feed both the covariance and the variance (at least `MIN_DOWN_OBS`).
- Months with a missing stock or market return are dropped (never treated as zero). If the minimum is not met, or the market variance is zero/missing, the beta is NaN – ordinary beta is never substituted for downside beta.
- Implemented with rolling sums, which gives the exact sample covariance/variance (the n−1 factors cancel in the ratio).

In [ ]:
def rolling_betas(excess_wide, window=WINDOW_MONTHS, min_down=MIN_DOWN_OBS, min_beta=MIN_BETA_OBS):
    mkt = factors["Mkt-RF"].reindex(excess_wide.index).to_numpy(float)[:, None]   # T x 1
    X = excess_wide.to_numpy(float)                                               # T x N
    Y = np.broadcast_to(mkt, X.shape)
    paired = ~np.isnan(X) & ~np.isnan(Y)
    out = {}
    for name, mask, min_n in [("beta", paired, min_beta), ("downside_beta", paired & (Y < 0), min_down)]:
        x, y = np.where(mask, X, 0.0), np.where(mask, Y, 0.0)
        roll = lambda a: pd.DataFrame(a).rolling(window, min_periods=window).sum().to_numpy()
        n, sx, sy, sxy, syy = roll(mask.astype(float)), roll(x), roll(y), roll(x * y), roll(y * y)
        with np.errstate(divide="ignore", invalid="ignore"):
            cov, var = sxy - sx * sy / n, syy - sy ** 2 / n
            beta = cov / var
        out[name] = np.where((n >= min_n) & (var > 1e-12), beta, np.nan)
        out["n_obs" if name == "beta" else "n_down_obs"] = n
    return out


def long_beta_table(est, window=WINDOW_MONTHS):
    # Audit table: one row per stock and estimation month with a downside beta.
    T, N = est["beta"].shape
    window_start = np.array([calendar[max(i - window + 1, 0)] for i in range(T)], dtype="datetime64[ns]")
    table = pd.DataFrame({
        "permno": np.tile(permnos, T),
        "estimation_month": np.repeat(calendar.to_numpy(), N),
        "downside_beta": est["downside_beta"].ravel(),
        "n_down_obs": est["n_down_obs"].ravel(),
        "beta": est["beta"].ravel(),
        "n_obs": est["n_obs"].ravel(),
        "window_start": np.repeat(window_start, N),
    })
    table["window_end"] = table["estimation_month"]
    return table.dropna(subset=["downside_beta"]).reset_index(drop=True)


def build_panel(window=WINDOW_MONTHS, min_down=MIN_DOWN_OBS, min_beta=MIN_BETA_OBS):
    # Stock-month panel: return in month t+1, betas estimated through month t.
    est = rolling_betas(excess_wide, window, min_down, min_beta)
    T, N = excess_wide.shape
    lag = lambda a: np.vstack([np.full((1, N), np.nan), a[:-1]])   # row t -> row t+1
    panel = pd.DataFrame({
        "return_month": np.repeat(calendar.to_numpy(), N),
        "permno": np.tile(permnos, T),
        "excess_return": excess_wide.to_numpy(float).ravel(),
        "beta": lag(est["beta"]).ravel(),
        "downside_beta": lag(est["downside_beta"]).ravel(),
        "n_down_obs": lag(est["n_down_obs"]).ravel(),
    })
    panel["estimation_month"] = panel["return_month"] - pd.offsets.MonthEnd(1)
    return panel, est


panel_raw, est_base = build_panel()
rolling_betas_long = long_beta_table(est_base)

# ---- Timing checks (no look-ahead) --------------------------------------------
valid = panel_raw.dropna(subset=["beta", "downside_beta"])
assert (valid["estimation_month"] < valid["return_month"]).all()
assert (valid["downside_beta"].notna() & (valid["n_down_obs"] >= MIN_DOWN_OBS)).all()
assert (est_base["n_down_obs"][~np.isnan(est_base["downside_beta"])] >= MIN_DOWN_OBS).all()

# Spot-check one estimate against a direct calculation (Cov/Var on the down-market months only)
row = rolling_betas_long.sample(1, random_state=1).iloc[0]
w = (excess_wide.index > row["window_start"] - pd.offsets.MonthEnd(1)) & (excess_wide.index <= row["estimation_month"])
xs, ys = excess_wide.loc[w, row["permno"]], factors.loc[w, "Mkt-RF"]
keep = xs.notna() & ys.lt(0)
direct = np.cov(xs[keep], ys[keep], ddof=1)[0, 1] / np.var(ys[keep], ddof=1)
print(f"Spot check PERMNO {int(row['permno'])}, {row['estimation_month']:%Y-%m}: "
      f"rolling = {row['downside_beta']:.6f}, direct = {direct:.6f}, down months = {int(row['n_down_obs'])}")
assert np.isclose(row["downside_beta"], direct)

print(f"\nRolling beta table: {len(rolling_betas_long):,} stock-months with a downside beta "
      f"(first estimate {rolling_betas_long['estimation_month'].min():%Y-%m}).")
display(rolling_betas_long.head())

## 5. Regression sample, winsorization and attrition
Dependent variable: next-month excess return. Regressors: ordinary beta and downside beta **as measured at the end of the previous month**. Observations missing any of the three are dropped, so CAPM, Model 1 and Model 2 are always estimated on exactly the same stock-months.

Winsorization: the dependent variable and both beta characteristics are winsorized **cross-sectionally within each month** at the 1st/99th percentiles (`WINSOR_LIMITS`) for the *regressions only*; the betas themselves were estimated from raw returns. The effect of 5/95 and no winsorization is shown in the sensitivity section.

In [ ]:
REG_COLS = ["excess_return", "beta", "downside_beta"]


def winsorize_by_month(df, cols, limits):
    df = df.copy()
    for c in cols:
        df[c] = df.groupby("return_month")[c].transform(
            lambda s: s.clip(lower=s.quantile(limits[0]), upper=s.quantile(limits[1]))
        )
    return df


def prepare(panel_raw, start=None, end=None, exclude=(), winsor=WINSOR_LIMITS):
    p = panel_raw.dropna(subset=REG_COLS)
    if start is not None:
        p = p[p["return_month"] >= pd.Timestamp(start)]
    if end is not None:
        p = p[p["return_month"] <= pd.Timestamp(end)]
    for a, b in exclude:
        p = p[~p["return_month"].between(pd.Timestamp(a), pd.Timestamp(b))]
    return winsorize_by_month(p, REG_COLS, winsor) if winsor else p.copy()


# ---- Sample attrition ------------------------------------------------------------
in_window = panel_raw.dropna(subset=["excess_return"])
if REG_START is not None:
    in_window = in_window[in_window["return_month"] >= pd.Timestamp(REG_START)]
if REG_END is not None:
    in_window = in_window[in_window["return_month"] <= pd.Timestamp(REG_END)]
first_beta_month = in_window.dropna(subset=["beta"])["return_month"].min()
in_window = in_window[in_window["return_month"] >= first_beta_month]
step2 = in_window.dropna(subset=["beta"])
step3 = step2.dropna(subset=["downside_beta"])
attrition = pd.DataFrame({
    "step": [f"Stock-months with a return, {first_beta_month:%Y-%m} onward (first month with a full 60-month window)",
             "Drop: no usable lagged ordinary beta (< MIN_BETA_OBS valid months)",
             "Drop: no usable lagged downside beta (< MIN_DOWN_OBS down-market months)",
             "Final regression sample"],
    "rows_affected": [len(in_window), len(in_window) - len(step2), len(step2) - len(step3), len(step3)],
    "rows_remaining": [len(in_window), len(step2), len(step3), len(step3)],
})
display(attrition)

# ---- Baseline sample with 1/99 winsorization ------------------------------------
panel = prepare(panel_raw, REG_START, REG_END)
pre = panel_raw.dropna(subset=REG_COLS)
pre = pre[pre["return_month"].isin(panel["return_month"].unique())]
changed = {c: int((winsorize_by_month(pre, [c], WINSOR_LIMITS)[c] != pre[c]).sum()) for c in REG_COLS}
print("Values changed by winsorization at", WINSOR_LIMITS, ":", changed)
print(f"Regression sample: {len(panel):,} stock-months, {panel['permno'].nunique()} securities, "
      f"{panel['return_month'].nunique()} months ({panel['return_month'].min():%Y-%m} to {panel['return_month'].max():%Y-%m})")

# ---- Descriptives ------------------------------------------------------------------
desc = panel[REG_COLS].describe(percentiles=[.05, .5, .95]).T
desc.loc["excess_return", ["mean", "std", "min", "5%", "50%", "95%", "max"]] *= 100
desc = desc.rename(index={"excess_return": "excess_return (% per month)"})
display(desc[["count", "mean", "std", "min", "5%", "50%", "95%", "max"]].round(3))

cs_corr = np.array([g["beta"].corr(g["downside_beta"]) for _, g in panel.groupby("return_month")])
print(f"Average cross-sectional correlation, beta vs downside beta: {cs_corr.mean():.3f}  "
      f"-> variance inflation factor = {1 / (1 - cs_corr.mean() ** 2):.2f}")
print("High collinearity inflates standard errors (not bias): judge gamma_D's t-stat in Model 2 with that in mind.")

## 6. Main regressions: CAPM benchmark, Model 1, Model 2
Each month's cross-sectional OLS gives one slope per regressor; the reported coefficient is the time-series mean of those slopes, with **Newey–West (6 lag)** standard errors and t-statistics (df = months − 1), which account for cross-sectional correlation (by construction) and serial correlation across months. Coefficients are in **% per month per unit of beta**. $R^2$ figures are the average cross-sectional $R^2$ over months.

In [ ]:
SPECS = {
    "CAPM benchmark": ["beta"],
    "Model 1: downside beta": ["downside_beta"],
    "Model 2: CAPM + downside": ["beta", "downside_beta"],
}
fm = {name: fama_macbeth(panel, regs) for name, regs in SPECS.items()}


def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""


def main_table(fm):
    terms = [("const", "Intercept"), ("beta", "Ordinary beta (CAPM)"), ("downside_beta", "Downside beta")]
    rows = []
    for term, label in terms:
        coef_row, t_row = {"": label}, {"": ""}
        for name, res in fm.items():
            s = res["summary"]
            if term in s.index:
                coef_row[name] = f"{100 * s.loc[term, 'coef']:.3f}{stars(s.loc[term, 'p'])}"
                t_row[name] = f"({s.loc[term, 't']:.2f})"
            else:
                coef_row[name], t_row[name] = "", ""
        rows += [coef_row, t_row]
    first = next(iter(fm.values()))
    for label, fmt in [("Avg cross-sectional R-squared", lambda r: f"{r['stats']['r2'].mean():.4f}"),
                       ("Avg cross-sectional adj. R-squared", lambda r: f"{r['stats']['adj_r2'].mean():.4f}"),
                       ("Avg stocks per month", lambda r: f"{r['stats']['n_stocks'].mean():.0f}"),
                       ("Months", lambda r: f"{len(r['gamma'])}")]:
        rows.append({"": label, **{n: fmt(r) for n, r in fm.items()}})
    return pd.DataFrame(rows).set_index("")


table_main = main_table(fm)
display(table_main)
print("Coefficients in % per month; Newey-West t-statistics (6 lags) in parentheses; * p<0.10, ** p<0.05, *** p<0.01.")
print(f"Sample: {panel['return_month'].min():%Y-%m} to {panel['return_month'].max():%Y-%m}.  "
      f"Average market excess return over the same months: {100 * factors.loc[panel['return_month'].unique(), 'Mkt-RF'].mean():.3f}% per month "
      "(CAPM predicts the ordinary-beta slope equals this and the intercept is zero).")

# ---- Full statistical output for the downside-beta coefficient --------------------
for name in ["Model 1: downside beta", "Model 2: CAPM + downside"]:
    s = fm[name]["summary"].loc["downside_beta"]
    verdict = "reject" if s["p"] < SIGNIFICANCE_LEVEL else "fail to reject"
    print(f"\n{name}: gamma_D = {100 * s['coef']:.3f}% per month, NW s.e. = {100 * s['se']:.3f}, "
          f"t = {s['t']:.2f}, p = {s['p']:.3f}, 95% CI [{100 * s['ci_low']:.3f}, {100 * s['ci_high']:.3f}] "
          f"-> {verdict} H0: gamma_D = 0 at 5%")

sd_down = panel.groupby("return_month")["downside_beta"].std().mean()
g2 = fm["Model 2: CAPM + downside"]["summary"].loc["downside_beta", "coef"]
print(f"\nEconomic size (Model 2): a 1-s.d. higher downside beta (s.d. = {sd_down:.2f}) is associated with "
      f"{100 * g2 * sd_down:.3f}% per month ({1200 * g2 * sd_down:.2f}% per year, simple annualization).")

# ---- Does downside beta add explanatory power beyond CAPM? ------------------------
d = fm["Model 2: CAPM + downside"]["stats"]["adj_r2"] - fm["CAPM benchmark"]["stats"]["adj_r2"]
b, se, t, p, df = ols_hac(d.to_numpy(), np.ones((len(d), 1)), NW_LAGS)
print(f"Adjusted R-squared gain, Model 2 minus CAPM: {b[0]:.5f} on average (NW t = {t[0]:.2f}, p = {p[0]:.3f}).")
d1 = fm["Model 1: downside beta"]["stats"]["adj_r2"] - fm["CAPM benchmark"]["stats"]["adj_r2"]
b1, se1, t1, p1, _ = ols_hac(d1.to_numpy(), np.ones((len(d1), 1)), NW_LAGS)
print(f"Adjusted R-squared gain, Model 1 minus CAPM: {b1[0]:.5f} on average (NW t = {t1[0]:.2f}, p = {p1[0]:.3f}).")
print("Reading these: an R-squared gain says downside beta helps describe *how stocks moved* in a given month (it is a second, noisy "
      "measure of market sensitivity), whereas whether downside beta is *rewarded* is the t-statistic on gamma_D above. The two can differ.")
table_main.to_csv(OUT_DIR / "table_main_fama_macbeth.csv")

### Cross-check: pooled OLS with standard errors clustered by month
Pooling all stock-months in one regression is *not* the primary test (the market-wide shocks in each month are common to all stocks, so conventional pooled standard errors would be far too small). Clustering by month fixes the cross-sectional dependence, but not serial correlation, so treat it as a robustness check on the Fama–MacBeth result.

In [ ]:
pooled_rows = []
for name, regs in SPECS.items():
    tab, r2, n = pooled_ols_cluster(panel, regs)
    for term in tab.index:
        pooled_rows.append({"model": name, "term": term, "coef (% per month)": 100 * tab.loc[term, "coef"],
                            "cluster s.e.": 100 * tab.loc[term, "se"], "t": tab.loc[term, "t"],
                            "p": tab.loc[term, "p"], "R2 (pooled)": r2, "N": n})
pooled_table = pd.DataFrame(pooled_rows)
display(pooled_table.round(4))
pooled_table.to_csv(OUT_DIR / "table_pooled_ols_clustered.csv", index=False)

## 7. Graphs of the fitted regressions
Raw stock-month scatters are dominated by noise (monthly stock returns have a ~10% standard deviation), so each figure shows (left) a random sample of raw points and (right) **binned averages** with their standard errors; the fitted line is the Fama–MacBeth regression (mean intercept and slope).

- **Figure 1 (Model 1):** excess return against downside beta, line = mean intercept + γ_D × downside beta.
- **Figure 2 (Model 2):** a *partial-regression* plot. Within each month, both next-month excess return and downside beta are first residualized on ordinary beta; the plot shows the residuals, and the line has slope γ_D from Model 2 (by the Frisch–Waugh result in the OLS slides, this is exactly the Model 2 downside-beta coefficient holding ordinary beta fixed).

In [ ]:
def binned(x, y, bins=30):
    q = pd.qcut(x, bins, duplicates="drop")
    g = pd.DataFrame({"x": x, "y": y}).groupby(q, observed=True)
    return g["x"].mean(), 100 * g["y"].mean(), 100 * g["y"].sem()


def regression_figure(x, y, intercept, slope, xlabel, ylabel, title, path, seed=0):
    # x in beta units, y in decimal return; intercept/slope in decimal-return units
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    sample = np.random.default_rng(seed).choice(len(x), size=min(20000, len(x)), replace=False)
    grid = np.linspace(np.quantile(x, 0.01), np.quantile(x, 0.99), 100)
    axes[0].scatter(x[sample], 100 * y[sample], s=3, alpha=0.08, color="tab:blue", rasterized=True)
    axes[0].plot(grid, 100 * (intercept + slope * grid), color="crimson", lw=2.2, label="Fama-MacBeth fit")
    axes[0].axhline(0, color="black", lw=0.8)
    axes[0].set_title("Raw stock-months (20,000 random points)")
    bx, by, bse = binned(x, y)
    axes[1].errorbar(bx, by, yerr=1.96 * bse, fmt="o", color="tab:blue", ecolor="lightsteelblue", capsize=2,
                     label="Binned mean (30 bins) ± 1.96 s.e.")
    axes[1].plot(grid, 100 * (intercept + slope * grid), color="crimson", lw=2.2, label="Fama-MacBeth fit")
    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].set_title("Binned averages")
    for ax in axes:
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.legend(loc="best", fontsize=8)
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


# ---- Figure 1: Model 1 ---------------------------------------------------------------
s1 = fm["Model 1: downside beta"]["summary"]
regression_figure(
    panel["downside_beta"].to_numpy(), panel["excess_return"].to_numpy(),
    s1.loc["const", "coef"], s1.loc["downside_beta", "coef"],
    "Downside beta (end of month t)", "Excess return in month t+1 (%)",
    f"Model 1: next-month excess return vs. downside beta   "
    f"(slope = {100 * s1.loc['downside_beta', 'coef']:.2f}% per unit, t = {s1.loc['downside_beta', 't']:.2f})",
    OUT_DIR / "fig1_model1_downside_beta.png",
)

# ---- Figure 2: Model 2 (partial regression on downside beta, ordinary beta held fixed) ----
xs, ys, slopes = [], [], []
for month, g in panel.groupby("return_month", sort=True):
    Z = np.column_stack([np.ones(len(g)), g["beta"].to_numpy(float)])
    resid = lambda v: v - Z @ np.linalg.lstsq(Z, v, rcond=None)[0]
    rx, ry = resid(g["downside_beta"].to_numpy(float)), resid(g["excess_return"].to_numpy(float))
    xs.append(rx)
    ys.append(ry)
    slopes.append((rx @ ry) / (rx @ rx))
xs, ys = np.concatenate(xs), np.concatenate(ys)
s2 = fm["Model 2: CAPM + downside"]["summary"]
assert np.isclose(np.mean(slopes), s2.loc["downside_beta", "coef"]), "FWL slope must equal Model 2 coefficient"
regression_figure(
    xs, ys, 0.0, s2.loc["downside_beta", "coef"],
    "Downside beta, residual after removing ordinary beta", "Excess return in month t+1, residual (%)",
    f"Model 2: partial regression, ordinary beta held fixed   "
    f"(slope = {100 * s2.loc['downside_beta', 'coef']:.2f}% per unit, t = {s2.loc['downside_beta', 't']:.2f})",
    OUT_DIR / "fig2_model2_partial_regression.png", seed=1,
)

## 8. Sensitivity analysis
The same three regressions are re-run under alternative assumptions, one change at a time:

| Alternative | What changes |
|---|---|
| Excluding 2008–09 / excluding 2020 | drops the global financial crisis / the COVID-19 crash months |
| 2005–2012 vs. 2013–2025 | early vs. late sample (downside-risk pricing was published in the mid-2000s, so the late sample is "after the factor became widely known") |
| Winsorization 5/95 and none | outlier treatment for returns and betas |
| 36-month beta window | betas estimated over 36 months (min. 24 total / 8 down-market months) instead of 60 |

The table reports $\gamma_D$ in Model 1 and Model 2, the CAPM slope, and the gain in adjusted $R^2$ over the CAPM benchmark. Compare significance and sign, not just magnitude.

In [ ]:
def scenario_row(label, p):
    capm = fama_macbeth(p, ["beta"])
    m1 = fama_macbeth(p, ["downside_beta"])
    m2 = fama_macbeth(p, ["beta", "downside_beta"])
    gain = m2["stats"]["adj_r2"] - capm["stats"]["adj_r2"]
    b, se, t, pv, _ = ols_hac(gain.to_numpy(), np.ones((len(gain), 1)), NW_LAGS)
    g = lambda r, term: (100 * r["summary"].loc[term, "coef"], r["summary"].loc[term, "t"], r["summary"].loc[term, "p"])
    cell = lambda v: f"{v[0]:.3f}{stars(v[2])} ({v[1]:.2f})"
    return {
        "Scenario": label,
        "Months": len(m2["gamma"]),
        "Stocks/month": round(m2["stats"]["n_stocks"].mean()),
        "CAPM: gamma_beta": cell(g(capm, "beta")),
        "Model 1: gamma_D": cell(g(m1, "downside_beta")),
        "Model 2: gamma_beta": cell(g(m2, "beta")),
        "Model 2: gamma_D": cell(g(m2, "downside_beta")),
        "Adj. R2 gain vs CAPM (pp)": f"{100 * b[0]:.3f}{stars(pv[0])} ({t[0]:.2f})",
    }


panel_36, _ = build_panel(window=36, min_down=8, min_beta=24)
scenarios = [
    ("Baseline (60m window, 1/99 winsor, full sample)", prepare(panel_raw, REG_START, REG_END)),
    ("Excluding 2008-2009", prepare(panel_raw, REG_START, REG_END, exclude=[("2008-01-31", "2009-12-31")])),
    ("Excluding 2020 (COVID)", prepare(panel_raw, REG_START, REG_END, exclude=[("2020-02-29", "2020-12-31")])),
    ("2005-2012 only", prepare(panel_raw, "2005-01-31", "2012-12-31")),
    ("2013-2025 only", prepare(panel_raw, "2013-01-31", REG_END)),
    ("Winsorize 5/95", prepare(panel_raw, REG_START, REG_END, winsor=(0.05, 0.95))),
    ("No winsorization", prepare(panel_raw, REG_START, REG_END, winsor=None)),
    ("36-month beta window", prepare(panel_36, REG_START, REG_END)),
]
sensitivity = pd.DataFrame([scenario_row(label, p) for label, p in scenarios]).set_index("Scenario")
display(sensitivity)
print("Cells: coefficient in % per month (Newey-West t-stat); adj. R2 gain in percentage points (fit, not pricing). * p<0.10, ** p<0.05, *** p<0.01.")
sensitivity.to_csv(OUT_DIR / "table_sensitivity.csv")

## 9. Bad times: when does the downside-beta slope pay?
A risk-based (rational) explanation requires that the premium is tied to bad states of the world; a behavioral/mispricing explanation does not. Using the monthly Model 2 slope series $\gamma_{D,t}$ (the return to one extra unit of downside beta in month *t*), we look at how it behaves (i) in down- vs. up-market months, (ii) in NBER recessions vs. expansions, and (iii) against the market excess return. NBER recession months in the sample: December 2007 – June 2009 and February – April 2020.

In [ ]:
gamma_d = fm["Model 2: CAPM + downside"]["gamma"]["downside_beta"] * 100      # % per month
mkt = factors.loc[gamma_d.index, "Mkt-RF"] * 100
recession = pd.Series(False, index=gamma_d.index)
for a, b_ in [("2007-12-31", "2009-06-30"), ("2020-02-29", "2020-04-30")]:
    recession |= gamma_d.index.to_series().between(pd.Timestamp(a), pd.Timestamp(b_))


def mean_row(label, mask):
    x = gamma_d[mask]
    se = x.std(ddof=1) / np.sqrt(len(x))
    return {"State": label, "Months": len(x), "Mean gamma_D (% per month)": x.mean(),
            "t (iid)": x.mean() / se if len(x) > 1 else np.nan}

states = pd.DataFrame([
    mean_row("Market down (Mkt-RF < 0)", mkt < 0), mean_row("Market up (Mkt-RF >= 0)", mkt >= 0),
    mean_row("NBER recession", recession), mean_row("Expansion", ~recession),
    mean_row("All months", pd.Series(True, index=gamma_d.index)),
]).set_index("State")
display(states.round(3))

# gamma_D,t = a + b * (Mkt-RF)_t + e  with Newey-West standard errors
X = np.column_stack([np.ones(len(mkt)), mkt.to_numpy()])
b, se, t, p, df = ols_hac(gamma_d.to_numpy(), X, NW_LAGS)
print(f"gamma_D,t on market excess return: slope = {b[1]:.3f} (NW t = {t[1]:.2f}, p = {p[1]:.3f}); intercept = {b[0]:.3f} (t = {t[0]:.2f}).")
print("gamma_D,t is a relative return (high vs. low downside beta, ordinary beta held fixed): a positive slope on Mkt-RF would mean the "
      "slope pays off in up-markets and loses in down-markets; a slope near zero means no systematic link to the market's direction.")
print("Months that matter most for the mean (% per month):  largest:",
      {d.strftime("%Y-%m"): round(v, 2) for d, v in gamma_d.nlargest(4).items()},
      " smallest:", {d.strftime("%Y-%m"): round(v, 2) for d, v in gamma_d.nsmallest(4).items()})
print("Few recession months (22) and a few extreme months dominate the recession average: judge its reliability with the t-stat, not the size.")
states.to_csv(OUT_DIR / "table_bad_times.csv")

## 10. Assumptions, limitations, reproducibility
- **Look-ahead:** betas dated *t* use only months ≤ *t* and predict returns in *t+1* (asserted in the code).
- **Returns:** supplied monthly `total_return` (dividends included); no price-based recomputation; no separate delisting-return field is available, so delisting returns are whatever the file reports.
- **Universe:** S&P 500 members only, in the months they were members – large, liquid stocks, which understates the role of small illiquid stocks and means that stocks enter the sample only after joining the index.
- **Equal weighting:** Fama–MacBeth treats each stock equally within a month; market caps are not in the data, so value-weighted tests are not possible.
- **Estimated regressors:** the betas are noisy estimates (errors-in-variables), which typically biases slope coefficients toward zero.
- **Inference:** Newey–West with 6 lags on the monthly slope series; the downside and ordinary betas are highly correlated, which inflates standard errors on both.
- **Outputs:** tables (CSV) and figures (PNG) are written to `./outputs`. Run with *Restart & Run All*; required packages: numpy, pandas, matplotlib (no SciPy / statsmodels).